# ICLR 2027: Neutral-Mode Exactness Audit — Final Targeted Test v1
## Geometry Determines Acceleration in Phase Retrieval
### Targeted Diagnostic for MUSAN Real-Audio Cases with `neutral_count_topk = 2`

**Role**: Student A (Official Targeted Audit Pipeline)  
**Target Files**:
1. `noise-free-sound-0043.wav`
2. `noise-free-sound-0389.wav`

**Mục tiêu Khoa học**:
Phân biệt rõ ràng ba khả năng toán học theo tài liệu chuẩn `Neutral_Mode_Exactness_Audit_Final_Targeted_Test_v1.md`:
- **Outcome A (Genuine 2D neutral symmetry)**: $\lambda_1 = 1, \lambda_2 = 1$.
- **Outcome B (Near-neutral transverse mode)**: $\lambda_1 = 1, \lambda_2 = 1 - \delta$ với $0 < \delta \ll 1$.
- **Outcome C (Numerically unresolved)**: Cận ngưỡng số học chưa phân biệt được tại độ phân giải hiện tại.

**Quy tắc bất biến (Strict Research Rules)**:
- **KHÔNG rerun toàn bộ 202 files.**
- **KHÔNG sửa core algorithm.**
- Đóng băng 100% cấu hình tiền xử lý, kích thước $N=2048$, STFT và toán tử Jacobian từ thí nghiệm gốc.
- Handoff đầy đủ `neutral_mode_exactness_audit.zip` cho hội đồng thẩm định.

In [ ]:
# [Cell 1] Hardware Check & Mathematical Foundations
# Self-contained isometric STFT analysis and Jacobian LinearOperator
import os, sys, math, time, json, csv, shutil, zipfile, platform, hashlib
from pathlib import Path
import numpy as np
import scipy
from scipy.io import wavfile
from scipy.sparse.linalg import LinearOperator, eigsh
import matplotlib.pyplot as plt

print("=== SYSTEM & HARDWARE INSPECTION ===")
print(f"Platform: {platform.platform()}")
print(f"Python:   {sys.version}")
print(f"NumPy:    {np.__version__}")
print(f"SciPy:    {scipy.__version__}")

ROOT = Path("YOUR_WORKING_PATH") if Path("YOUR_WORKING_PATH").exists() else Path(".").resolve()
OUT_DIR = ROOT / "results" / "neutral_mode_exactness_audit"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nWorking Root:     {ROOT}")
print(f"Output Directory: {OUT_DIR}")

# --------------------------------------------------------------------------
# FROZEN MATHEMATICAL ENGINE: STFT & MATRIX-FREE JACOBIAN
# --------------------------------------------------------------------------
def hann_window(length: int) -> np.ndarray:
    """Periodic Hann window, exactly matching the 202-file experiment."""
    n = np.arange(length)
    return 0.5 - 0.5 * np.cos(2 * np.pi * n / length)

class FFTAnalysisOperator:
    def __init__(self, signal_length: int, window: np.ndarray, hop: int, n_fft: int):
        self.N = int(signal_length)
        self.window = np.asarray(window, dtype=float)
        self.W = int(len(self.window))
        self.hop = int(hop)
        self.n_fft = int(n_fft)
        if self.N % self.hop != 0:
            raise ValueError("signal_length must be divisible by hop")
        if self.n_fft < self.W:
            raise ValueError("n_fft must be >= window length")
        self.n_frames = self.N // self.hop
        self.M = self.n_frames * self.n_fft
        diag = np.zeros(self.N, dtype=float)
        for m in range(self.n_frames):
            idx = (m * self.hop + np.arange(self.W)) % self.N
            diag[idx] += self.n_fft * self.window**2
        if np.min(diag) <= 1e-14:
            raise ValueError("STFT frame operator is singular")
        self.frame_diag = diag
        self.frame_sqrt = np.sqrt(diag)
        self.frame_inv_sqrt = 1.0 / self.frame_sqrt

    @property
    def signal_dim(self) -> int:
        return self.N

    @property
    def coeff_dim(self) -> int:
        return self.M

    def analyze(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=np.complex128)
        out = np.empty((self.n_frames, self.n_fft), dtype=np.complex128)
        for m in range(self.n_frames):
            idx = (m * self.hop + np.arange(self.W)) % self.N
            frame = np.zeros(self.n_fft, dtype=np.complex128)
            frame[: self.W] = self.window * x[idx]
            out[m] = np.fft.fft(frame)
        return out.reshape(-1)

    def adjoint(self, c: np.ndarray) -> np.ndarray:
        C = np.asarray(c, dtype=np.complex128).reshape(self.n_frames, self.n_fft)
        out = np.zeros(self.N, dtype=np.complex128)
        for m in range(self.n_frames):
            idx = (m * self.hop + np.arange(self.W)) % self.N
            frame = np.fft.ifft(C[m]) * self.n_fft
            out[idx] += self.window * frame[: self.W]
        return out

    def synthesize_ls(self, c: np.ndarray) -> np.ndarray:
        return self.adjoint(c) / self.frame_diag

    def q_analyze(self, z: np.ndarray) -> np.ndarray:
        return self.analyze(np.asarray(z, dtype=np.complex128) * self.frame_inv_sqrt)

    def q_adjoint(self, c: np.ndarray) -> np.ndarray:
        return self.adjoint(c) * self.frame_inv_sqrt

def c2r(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=np.complex128)
    return np.concatenate([z.real, z.imag])

def r2c(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=float)
    n = v.size // 2
    return v[:n] + 1j * v[n:]

def tangent_project(c_star: np.ndarray, h: np.ndarray, zero_tol: float = 1e-14) -> np.ndarray:
    c = np.asarray(c_star, dtype=np.complex128)
    h = np.asarray(h, dtype=np.complex128)
    mag = np.abs(c)
    if np.min(mag) <= zero_tol:
        raise ValueError("regular tangent projection requires nonzero coefficients")
    u = c / mag
    return h - u * np.real(np.conj(u) * h)

def jacobian_signal_action(c_star: np.ndarray, op, z: np.ndarray) -> np.ndarray:
    qz = op.q_analyze(z)
    return op.q_adjoint(tangent_project(c_star, qz))

def jacobian_linear_operator(c_star: np.ndarray, op) -> LinearOperator:
    n = op.signal_dim
    def mv(v):
        z = r2c(v)
        return c2r(jacobian_signal_action(c_star, op, z))
    return LinearOperator((2*n, 2*n), matvec=mv, rmatvec=mv, dtype=float)

print("[OK] Core mathematical engine initialized successfully.")


In [ ]:
# [Cell 2] Target File Discovery & Preprocessing
# Locates noise-free-sound-0043.wav and noise-free-sound-0389.wav
# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU CỦA BẠN TRÊN KAGGLE
# ==============================================================================
path = YOUR_PATH = "YOUR_DATA_PATH"

TARGET_FILENAMES = ["noise-free-sound-0043.wav", "noise-free-sound-0389.wav"]

CANDIDATE_SEARCH_PATHS = [
    Path(YOUR_PATH),
    Path("YOUR_DATA_PATH"),
    ROOT,
    ROOT / "musan",
    ROOT / "data" / "musan",
    Path("YOUR_PROJECT_PATH"),
]

def locate_target_files(filenames, search_roots):
    found = {}
    for fn in filenames:
        for s_root in search_roots:
            if not s_root.exists():
                continue
            # Direct check
            matches = list(s_root.rglob(fn))
            if matches:
                found[fn] = matches[0]
                break
    return found

print("Searching for target audit files...")
located_files = locate_target_files(TARGET_FILENAMES, CANDIDATE_SEARCH_PATHS)
for fn in TARGET_FILENAMES:
    if fn in located_files:
        print(f"  [FOUND] {fn} -> {located_files[fn]}")
    else:
        print(f"  [MISSING] {fn} not found in searched paths.")

if len(located_files) < len(TARGET_FILENAMES):
    print("\n[CHÚ Ý] Nếu file chưa được tìm thấy, vui lòng kiểm tra lại biến 'YOUR_PATH' ở đầu cell!")

# Audio reading and excerpt extraction (exactly matching original run)
def _mono_float(p: Path) -> tuple[int, np.ndarray]:
    fs, raw = wavfile.read(p)
    raw = np.asarray(raw)
    dtype = raw.dtype
    if raw.ndim > 1:
        x = np.mean(raw.astype(float), axis=1)
    else:
        x = raw.astype(float)
    if np.issubdtype(dtype, np.integer):
        mx = np.iinfo(dtype).max
        x = x / max(float(mx), 1.0)
    x = x - np.mean(x)
    norm = np.linalg.norm(x)
    return int(fs), x / norm if norm > 0 else x

def _excerpt(x: np.ndarray, N: int) -> np.ndarray:
    if len(x) >= N:
        start = (len(x) - N) // 2
        return x[start : start + N]
    y = np.zeros(N, float)
    y[: len(x)] = x
    return y

N = 2048
op = FFTAnalysisOperator(N, hann_window(256), 32, 256)
print(f"STFT Operator initialized: N={N}, window=Hann(256), hop=32, n_fft=256")


In [ ]:
# [Cell 3] Test A --- High-Precision Top-Eigenvalue Extraction
# Tolerance: eig_tol = 1e-12, k >= 8 eigenvalues
# Quantities: lambda_j, delta_j = 1 - lambda_j, eigenpair residual r_j

K_EIGS = 8
PRIMARY_TOL = 1e-12

target_data = {}
top_eigenvalues_rows = []

for fn, fpath in located_files.items():
    print(f"\n{'='*60}")
    print(f"Processing File: {fn}")
    print(f"{'='*60}")
    
    fs, x = _mono_float(fpath)
    x_sub = _excerpt(x, N).astype(np.complex128)
    norm_x = np.linalg.norm(x_sub)
    x_sub /= max(norm_x, 1e-15)
    
    cstar = op.analyze(x_sub)
    A = jacobian_linear_operator(cstar, op)
    
    print(f"Computing top {K_EIGS} eigenvalues with eig_tol={PRIMARY_TOL:.0e}...")
    t0 = time.time()
    vals, vecs = eigsh(A, k=K_EIGS, which="LA", tol=PRIMARY_TOL, maxiter=10000)
    elapsed = time.time() - t0
    
    order = np.argsort(vals)[::-1]
    vals = np.clip(vals[order], -1e-12, 1.0 + 1e-12)
    vecs = vecs[:, order]
    
    target_data[fn] = {
        "path": fpath,
        "fs": fs,
        "x_sub": x_sub,
        "cstar": cstar,
        "A": A,
        "vals": vals,
        "vecs": vecs,
    }
    
    print(f"Solve time: {elapsed:.2f}s")
    print(f"Rank | {'Eigenvalue':<20} | {'1 - Eigenvalue (delta)':<22} | {'Solver Residual':<16} | Status")
    print("-" * 80)
    
    for rank in range(1, K_EIGS + 1):
        val = float(vals[rank - 1])
        vec = vecs[:, rank - 1]
        delta = float(1.0 - val)
        
        # Residual r_j = ||J v_j - lambda_j v_j|| / max(1, ||J v_j||)
        jv = A.matvec(vec)
        res_vec = jv - val * vec
        norm_res = float(np.linalg.norm(res_vec))
        norm_jv = float(np.linalg.norm(jv))
        r_j = norm_res / max(1.0, norm_jv)
        status = "CONVERGED" if r_j <= 1e-9 else "ACHIEVED_TOL"
        
        print(f"{rank:4d} | {val:<20.14f} | {delta:<22.14e} | {r_j:<16.2e} | {status}")
        
        top_eigenvalues_rows.append({
            "file_id": fn,
            "rank": rank,
            "eigenvalue": val,
            "one_minus_eigenvalue": delta,
            "eig_tol": PRIMARY_TOL,
            "solver_residual": r_j,
            "solver_status": status,
        })

# Save Test A outputs
with open(OUT_DIR / "top_eigenvalues.json", "w", encoding="utf-8") as f:
    json.dump(top_eigenvalues_rows, f, indent=2)

with open(OUT_DIR / "top_eigenvalues.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "rank", "eigenvalue", "one_minus_eigenvalue", "eig_tol", "solver_residual", "solver_status"])
    writer.writeheader()
    writer.writerows(top_eigenvalues_rows)

print(f"\n[OK] Test A complete. Saved {OUT_DIR / 'top_eigenvalues.csv'}")


In [ ]:
# [Cell 4] Test B --- Neutral-Count Tolerance Sweep
# Sweeps phase_tol in {1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10}
# Diagnostics: Check whether neutral_count transitions 2 -> 1

SWEEP_TOLS = [1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10]
neutral_sweep_rows = []

print("=== Test B: Neutral-Count Tolerance Sweep ===")
for fn, d in target_data.items():
    vals = d["vals"]
    print(f"\nFile: {fn}")
    print(f"{'phase_tol':<12} | {'neutral_count':<14} | {'lambda_plus (transverse)':<25} | {'1 - lambda_plus':<16}")
    print("-" * 72)
    
    for tol in SWEEP_TOLS:
        neutral = (vals >= 1.0 - tol)
        nc = int(np.sum(neutral))
        transverse = vals[~neutral]
        if len(transverse) > 0:
            lp = float(transverse[0])
            gap = float(1.0 - lp)
        else:
            lp = None
            gap = None
        
        lp_str = f"{lp:.14f}" if lp is not None else "N/A"
        gap_str = f"{gap:.6e}" if gap is not None else "N/A"
        print(f"{tol:<12.0e} | {nc:<14d} | {lp_str:<25} | {gap_str:<16}")
        
        neutral_sweep_rows.append({
            "file_id": fn,
            "phase_tol": tol,
            "neutral_count": nc,
            "lambda_plus_after_neutral_removal": lp,
        })

# Save Test B outputs
with open(OUT_DIR / "neutral_tolerance_sweep.json", "w", encoding="utf-8") as f:
    json.dump(neutral_sweep_rows, f, indent=2)

with open(OUT_DIR / "neutral_tolerance_sweep.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "phase_tol", "neutral_count", "lambda_plus_after_neutral_removal"])
    writer.writeheader()
    writer.writerows(neutral_sweep_rows)

print(f"\n[OK] Test B complete. Saved {OUT_DIR / 'neutral_tolerance_sweep.csv'}")


In [ ]:
# [Cell 5] Test C --- Solver Stability
# Evaluates eigensolve at eig_tol = 1e-8, 1e-10, 1e-12
# Quantifies ||lambda_j^(a) - lambda_j^(b)|| across resolutions

STABILITY_TOLS = [1e-8, 1e-10, 1e-12]
stability_rows = []

print("=== Test C: Solver Stability Across Resolutions ===")
for fn, d in target_data.items():
    A = d["A"]
    print(f"\nFile: {fn}")
    
    runs = {}
    for tol in STABILITY_TOLS:
        t_v, _ = eigsh(A, k=K_EIGS, which="LA", tol=tol, maxiter=10000)
        t_v = np.sort(np.clip(t_v, -1e-12, 1.0 + 1e-12))[::-1]
        runs[tol] = t_v
        print(f"  eig_tol={tol:.0e} -> lambda_1={t_v[0]:.14f}, lambda_2={t_v[1]:.14f}, lambda_3={t_v[2]:.14f}")
        
    pairs = [(1e-8, 1e-10), (1e-10, 1e-12)]
    for (tol_a, tol_b) in pairs:
        d_l1 = float(abs(runs[tol_a][0] - runs[tol_b][0]))
        d_l2 = float(abs(runs[tol_a][1] - runs[tol_b][1]))
        d_l3 = float(abs(runs[tol_a][2] - runs[tol_b][2]))
        
        pair_name = f"{tol_a:.0e}_vs_{tol_b:.0e}"
        print(f"  Difference ({pair_name}): |d_l1|={d_l1:.2e}, |d_l2|={d_l2:.2e}, |d_l3|={d_l3:.2e}")
        
        stability_rows.append({
            "file_id": fn,
            "eig_tol_pair": pair_name,
            "delta_lambda_1": d_l1,
            "delta_lambda_2": d_l2,
            "delta_lambda_3": d_l3,
        })

# Save Test C outputs
with open(OUT_DIR / "solver_stability.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "eig_tol_pair", "delta_lambda_1", "delta_lambda_2", "delta_lambda_3"])
    writer.writeheader()
    writer.writerows(stability_rows)

print(f"\n[OK] Test C complete. Saved {OUT_DIR / 'solver_stability.csv'}")


In [ ]:
# [Cell 6] Test D --- Global-Phase Direction Verification
# Evaluates real-Hilbert alignment with z_phase = i * c_star
# a_1 = |<v_1, z_phase>| / (||v_1|| ||z_phase||)
# a_E2 = ||P_E2 z_phase|| / ||z_phase|| for 2D subspace E2 = span{v_1, v_2}

phase_alignment_rows = []

print("=== Test D: Global-Phase Direction Verification ===")
for fn, d in target_data.items():
    x_sub = d["x_sub"]
    vecs = d["vecs"]
    
    # In isometric signal coordinates: z_phase = 1j * x_sub * op.frame_sqrt
    z_ph = 1j * x_sub * op.frame_sqrt
    norm_ph = np.linalg.norm(z_ph)
    z_ph /= max(norm_ph, 1e-15)
    v_ph = c2r(z_ph)
    v_ph /= np.linalg.norm(v_ph)
    
    v1 = vecs[:, 0]
    v2 = vecs[:, 1]
    
    inner_1 = float(np.dot(v1, v_ph))
    inner_2 = float(np.dot(v2, v_ph))
    a_1 = abs(inner_1)
    a_E2 = float(np.sqrt(inner_1**2 + inner_2**2))
    
    print(f"\nFile: {fn}")
    print(f"  Leading eigenvector alignment a_1:       {a_1:.8f}")
    print(f"  2D Eigenspace subspace alignment a_E2:  {a_E2:.8f}")
    print(f"  Projection weights: <v1, z_ph>={inner_1:+.6f}, <v2, z_ph>={inner_2:+.6f}")
    
    phase_alignment_rows.append({
        "file_id": fn,
        "a_1": a_1,
        "a_E2": a_E2,
        "v1_inner_phase": inner_1,
        "v2_inner_phase": inner_2,
    })

# Save Test D outputs
with open(OUT_DIR / "phase_alignment.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "a_1", "a_E2", "v1_inner_phase", "v2_inner_phase"])
    writer.writeheader()
    writer.writerows(phase_alignment_rows)

print(f"\n[OK] Test D complete. Saved {OUT_DIR / 'phase_alignment.csv'}")


In [ ]:
# [Cell 7] Decision Rules & Publication Diagnostic Figures
# Generates Waveform Energy, STFT Support, Eigenvector v2, and Tolerance Sweep plots
from IPython.display import Image, display

audit_verdicts = {}

for fn, d in target_data.items():
    vals = d["vals"]
    vecs = d["vecs"]
    x_sub = d["x_sub"]
    cstar = d["cstar"]
    
    # Get neutral counts from Test B
    nc_1e7 = [r["neutral_count"] for r in neutral_sweep_rows if r["file_id"] == fn and r["phase_tol"] == 1e-7][0]
    nc_1e10 = [r["neutral_count"] for r in neutral_sweep_rows if r["file_id"] == fn and r["phase_tol"] == 1e-10][0]
    a_E2 = [r["a_E2"] for r in phase_alignment_rows if r["file_id"] == fn][0]
    
    delta_1 = 1.0 - vals[0]
    delta_2 = 1.0 - vals[1]
    delta_3 = 1.0 - vals[2]
    
    # Official Decision Logic from Section 7
    if (nc_1e7 == 2 and nc_1e10 == 1) or (delta_2 > 1e-9 and a_E2 > 0.99):
        verdict = "OUTCOME_B_NEAR_NEUTRAL_TRANSVERSE_MODE"
        note = "The apparent second neutral mode at phase_tol = 1e-7 is an extremely slow transverse mode rather than a second exact phase symmetry."
        struct_req = False
    elif delta_2 < 1e-12 and a_E2 > 0.99:
        verdict = "OUTCOME_A_GENUINE_ADDITIONAL_NEUTRAL_DIRECTION"
        note = "The case exhibits an additional neutral direction and therefore lies outside the simple-neutral-mode regularity assumption."
        struct_req = True
    else:
        verdict = "OUTCOME_C_NUMERICALLY_UNRESOLVED"
        note = "Precision cannot resolve lambda_2 = 1 vs lambda_2 = 1 - delta."
        struct_req = True
        
    audit_verdicts[fn] = {
        "verdict": verdict,
        "note": note,
        "struct_req": struct_req,
        "nc_1e7": nc_1e7,
        "nc_1e10": nc_1e10,
        "delta_1": delta_1,
        "delta_2": delta_2,
        "delta_3": delta_3,
        "lambda_1": vals[0],
        "lambda_2": vals[1],
        "lambda_3": vals[2],
    }
    
    # --------------------------------------------------------------------------
    # 4-Panel Diagnostic Figure for each case
    # --------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Panel 1: Waveform Energy vs Time
    t_axis = np.arange(len(x_sub))
    energy = np.abs(x_sub)**2
    axes[0, 0].plot(t_axis, energy, color="#1f77b4", lw=1.2)
    axes[0, 0].set_title(f"{fn}: Waveform Energy vs Time", fontsize=11, fontweight="bold")
    axes[0, 0].set_xlabel("Sample Index t")
    axes[0, 0].set_ylabel("Instantaneous Energy |x(t)|^2")
    axes[0, 0].grid(True, alpha=0.3)
    
    # Panel 2: STFT Magnitude Heatmap (Time-Frequency)
    C_mat = np.abs(cstar.reshape(op.n_frames, op.n_fft)[:, :op.n_fft//2 + 1].T)
    im1 = axes[0, 1].imshow(20 * np.log10(np.maximum(C_mat, 1e-6)), aspect="auto", origin="lower", cmap="magma")
    axes[0, 1].set_title(f"{fn}: STFT Magnitude Spectrogram (dB)", fontsize=11, fontweight="bold")
    axes[0, 1].set_xlabel("Time Frame m")
    axes[0, 1].set_ylabel("Frequency Bin k")
    fig.colorbar(im1, ax=axes[0, 1], label="Magnitude (dB)")
    
    # Panel 3: Second Eigenvector v2 Time-Frequency Representation
    z2 = r2c(vecs[:, 1])
    q_z2 = op.q_analyze(z2)
    Q2_mat = np.abs(q_z2.reshape(op.n_frames, op.n_fft)[:, :op.n_fft//2 + 1].T)
    im2 = axes[1, 0].imshow(Q2_mat, aspect="auto", origin="lower", cmap="viridis")
    axes[1, 0].set_title(f"{fn}: Eigenvector v_2 Coordinate Map (1 - lambda_2 = {delta_2:.2e})", fontsize=11, fontweight="bold")
    axes[1, 0].set_xlabel("Time Frame m")
    axes[1, 0].set_ylabel("Frequency Bin k")
    fig.colorbar(im2, ax=axes[1, 0], label="|Q v_2|")
    
    # Panel 4: Neutral-Count vs Tolerance Sweep
    tols = [r["phase_tol"] for r in neutral_sweep_rows if r["file_id"] == fn]
    ncs = [r["neutral_count"] for r in neutral_sweep_rows if r["file_id"] == fn]
    axes[1, 1].plot([-np.log10(t) for t in tols], ncs, marker="o", lw=2, color="#d62728")
    axes[1, 1].axhline(1, color="green", ls="--", alpha=0.7, label="Target 1D Gauge Symmetry")
    axes[1, 1].set_title(f"{fn}: Neutral Mode Count vs -log10(phase_tol)", fontsize=11, fontweight="bold")
    axes[1, 1].set_xlabel("-log10(phase_tol)")
    axes[1, 1].set_ylabel("neutral_count")
    axes[1, 1].set_yticks([0, 1, 2, 3])
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    fig_path = FIG_DIR / f"diagnostic_{fn.replace('.wav', '')}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close()
    print(f"Saved diagnostic figure: {fig_path}")

# Display figures inline
for fn in target_data.keys():
    fig_path = FIG_DIR / f"diagnostic_{fn.replace('.wav', '')}.png"
    if fig_path.exists():
        display(Image(filename=str(fig_path)))


In [ ]:
# [Cell 8] Required Final Table (Section 9 Compliance)
# Formatted identically to the official ICLR 2027 protocol specification
from IPython.display import Markdown, display

table_md = ["| File | 1 - lambda_1 | 1 - lambda_2 | 1 - lambda_3 | neutral @ 10^-7 | neutral @ 10^-10 | phase alignment (a_1 / a_E2) | Final Verdict |"]
table_md.append("| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |")

for fn, v in audit_verdicts.items():
    a_info = [r for r in phase_alignment_rows if r["file_id"] == fn][0]
    align_str = f"{a_info['a_1']:.4f} / {a_info['a_E2']:.4f}"
    table_md.append(
        f"| `{fn}` | `{v['delta_1']:.4e}` | `{v['delta_2']:.4e}` | `{v['delta_3']:.4e}` | "
        f"`{v['nc_1e7']}` | `{v['nc_1e10']}` | `{align_str}` | **`{v['verdict']}`** |"
    )

full_table = "\n".join(table_md)
print("=== REQUIRED FINAL TABLE (SECTION 9) ===\n")
display(Markdown(full_table))

# Also save table as Markdown and CSV
with open(OUT_DIR / "final_table.md", "w", encoding="utf-8") as f:
    f.write(full_table + "\n")


In [ ]:
# [Cell 9] Packaging Supervisor Handoff Archive (Section 10 & 11 Compliance)
# Packages neutral_mode_exactness_audit.zip
import zipfile, datetime

# 1. Create summary.json
summary_list = []
for fn, v in audit_verdicts.items():
    a_info = [r for r in phase_alignment_rows if r["file_id"] == fn][0]
    summary_list.append({
        "file_id": fn,
        "verdict": v["verdict"],
        "lambda_1": v["lambda_1"],
        "lambda_2": v["lambda_2"],
        "lambda_3": v["lambda_3"],
        "gap_1": v["delta_1"],
        "gap_2": v["delta_2"],
        "gap_3": v["delta_3"],
        "neutral_count_at_1e-7": v["nc_1e7"],
        "neutral_count_at_1e-10": v["nc_1e10"],
        "phase_alignment": a_info["a_1"],
        "eigenspace_phase_alignment": a_info["a_E2"],
        "structural_test_required": v["struct_req"],
        "notes": v["note"],
    })

with open(OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_list, f, indent=2)

# 2. Create environment.txt & commands.txt
with open(OUT_DIR / "environment.txt", "w", encoding="utf-8") as f:
    f.write(f"Date: {datetime.datetime.now().isoformat()}\n")
    f.write(f"Platform: {platform.platform()}\n")
    f.write(f"Python: {sys.version}\n")
    f.write(f"NumPy: {np.__version__}\n")
    f.write(f"SciPy: {scipy.__version__}\n")

with open(OUT_DIR / "commands.txt", "w", encoding="utf-8") as f:
    f.write("# Reproduction command:\n")
    f.write("jupyter nbconvert --to notebook --execute Student_A_ICLR2027_Kaggle.ipynb\n")

# 3. Create README.md
readme_content = f"""# Neutral-Mode Exactness Audit --- Final Targeted Test v1
## ICLR 2027: Geometry Determines Acceleration in Phase Retrieval

### Overview
Targeted diagnostic package auditing the two MUSAN real-audio cases exhibiting `neutral_count_topk = 2` at `phase_tol = 1e-7`:
- `noise-free-sound-0043.wav`
- `noise-free-sound-0389.wav`

### Summary of Findings
"""
for item in summary_list:
    readme_content += f"- **{item['file_id']}**: Verdict `{item['verdict']}`. Gap 1 = {item['gap_1']:.2e}, Gap 2 = {item['gap_2']:.2e}. Subspace phase alignment = {item['eigenspace_phase_alignment']:.4f}.\n"

with open(OUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

# 4. Package into ZIP archive with POSIX paths
zip_path = ROOT / "neutral_mode_exactness_audit.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            rel_path = p.relative_to(OUT_DIR)
            z.write(p, arcname=rel_path.as_posix())

# Also copy to YOUR_WORKING_PATH/ if applicable
if Path("YOUR_WORKING_PATH").exists() and zip_path != Path("YOUR_WORKING_PATH/neutral_mode_exactness_audit.zip"):
    shutil.copy2(zip_path, "YOUR_WORKING_PATH/neutral_mode_exactness_audit.zip")

size_kb = zip_path.stat().st_size / 1024
hasher = hashlib.sha256()
hasher.update(zip_path.read_bytes())
sha = hasher.hexdigest()

print("=" * 70)
print(">>> AUDIT HANDOFF ARCHIVE SUCCESSFULLY CREATED <<<")
print(f"  Archive Path: {zip_path}")
print(f"  Archive Size: {size_kb:.1f} KB")
print(f"  SHA-256:      {sha}")
print("=" * 70)
print("\n[STOP RULE ACTIVATED] Tests A-D complete. Package ready for supervisor review.")
